# 04 · Preparación y exportación de la tabla minable

En este notebook limpiamos y transformamos los datos para obtener un CSV sin valores nulos y con una fila por cliente.

Primero debemos ejecutar el notebook 01. En esta práctica preparamos la tabla, pero no entrenamos un modelo.

Los cálculos de imputación y escalado se ajustan con toda la muestra para preparar el CSV. Para evaluar un modelo, habría que ajustarlos solo con los datos de entrenamiento.

## 1. Librerías y rutas

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 30)
sns.set_theme(style='whitegrid')

def localizar_proyecto():
    # Funciona al abrir Jupyter desde la carpeta principal o desde notebooks/.
    for ruta in [Path.cwd(), *Path.cwd().parents]:
        if (ruta / 'datos' / 'Caso Berka.zip').is_file():
            return ruta
    raise FileNotFoundError(
        'No se encontró datos/Caso Berka.zip. Abre Jupyter desde Berka_Equipo/.'
    )

RAIZ = localizar_proyecto()
INTERMEDIOS = RAIZ / 'intermedios'
RESULTADOS = RAIZ / 'resultados'
GRAFICOS = RESULTADOS / 'graficos'
for carpeta in [INTERMEDIOS, RESULTADOS, GRAFICOS]:
    carpeta.mkdir(parents=True, exist_ok=True)
print('Proyecto:', RAIZ.resolve())

from sklearn.preprocessing import StandardScaler

Proyecto: /mnt/data/Berka_Equipo


## 2. Cargar datos y seleccionar predictores explicados en el notebook 03

In [2]:
ruta=INTERMEDIOS/'01_base_clientes.csv'
if not ruta.is_file(): raise FileNotFoundError('Ejecuta primero el notebook 01.')
df = pd.read_csv(ruta)
NUMERICAS = ['edad_1998','antiguedad_cuenta_anios','num_transacciones',
            'importe_ingresos','importe_egresos',
            'importe_medio_transaccion','saldo_promedio']
CATEGORICAS = ['tipo_disposicion','frecuencia']
PREDICTORAS = NUMERICAS+CATEGORICAS
ETIQUETA='tiene_tarjeta'
assert df['client_id'].is_unique
print('Dataframe inicial:',df.shape)
print('Variables seleccionadas:',PREDICTORAS)

Dataframe inicial: (5369, 22)
Variables seleccionadas: ['edad_1998', 'antiguedad_cuenta_anios', 'num_transacciones', 'importe_ingresos', 'importe_egresos', 'importe_medio_transaccion', 'saldo_promedio', 'tipo_disposicion', 'frecuencia']


## 3. Eliminar columnas con más del 60% de valores nulos
Revisamos los valores faltantes en toda la tabla. La fecha de la primera tarjeta suele estar vacía cuando el cliente no tiene una tarjeta registrada, por lo que la dejamos fuera de los predictores.

In [3]:
porcentaje_nulos = 100*df.isna().mean()
columnas_descartadas = porcentaje_nulos[porcentaje_nulos>60].index.tolist()
print('Nulos (%) por columna:')
print(porcentaje_nulos.sort_values(ascending=False).round(2).to_string())
print('Columnas con >60% nulos:',columnas_descartadas)
assert ETIQUETA not in columnas_descartadas, 'La variable respuesta no puede descartarse.'
df_limpio = df.drop(columns=columnas_descartadas)
assert set(PREDICTORAS).issubset(df_limpio.columns), 'Revisar predictores descartados.'

Nulos (%) por columna:
fecha_primera_tarjeta        83.39
client_id                     0.00
district_id_cliente           0.00
sexo                          0.00
fecha_nacimiento              0.00
birth_number                  0.00
edad_1998                     0.00
disp_id                       0.00
tipo_disposicion              0.00
account_id                    0.00
frecuencia                    0.00
fecha_apertura_codificada     0.00
fecha_apertura                0.00
district_id_cuenta            0.00
antiguedad_cuenta_anios       0.00
n_tarjetas                    0.00
num_transacciones             0.00
importe_ingresos              0.00
importe_egresos               0.00
importe_medio_transaccion     0.00
saldo_promedio                0.00
tiene_tarjeta                 0.00
Columnas con >60% nulos: ['fecha_primera_tarjeta']


## 4. Valores atípicos con el criterio del rango intercuartílico (RIC)
Usamos los límites Q1 − 1.5×RIC y Q3 + 1.5×RIC para identificar valores atípicos. No los eliminamos automáticamente porque un valor alto también puede ser correcto.

In [4]:
filas_outliers=[]
for var in NUMERICAS:
    s=df_limpio[var].dropna()
    q1,q3=s.quantile([.25,.75]); ric=q3-q1
    inferior=q1-1.5*ric; superior=q3+1.5*ric
    n=int(((s<inferior)|(s>superior)).sum())
    filas_outliers.append({
        'variable':var,'limite_inferior':inferior,'limite_superior':superior,
        'numero_atipicos':n,'porcentaje':round(100*n/len(s),2),
        'decision':'conservar: validar con negocio antes de excluir'})
resumen_outliers=pd.DataFrame(filas_outliers)
display(resumen_outliers.round(2))
resumen_outliers.to_csv(RESULTADOS/'04_diagnostico_outliers.csv',
                       index=False,encoding='utf-8-sig')

,variable,limite_inferior,limite_superior,numero_atipicos,porcentaje,decision
0,edad_1998,-12.00,100.00,0,0.00,conservar: validar con negocio antes de excluir
1,antiguedad_cuenta_anios,-2.15,9.34,0,0.00,conservar: validar con negocio antes de excluir
2,num_transacciones,-162.50,625.50,15,0.28,conservar: validar con negocio antes de excluir
3,importe_ingresos,-933983.90,2156195.30,262,4.88,conservar: validar con negocio antes de excluir
4,importe_egresos,-924780.50,2061409.10,274,5.10,conservar: validar con negocio antes de excluir
5,importe_medio_transaccion,-6728.96,17273.33,38,0.71,conservar: validar con negocio antes de excluir
6,saldo_promedio,-13519.55,84327.63,0,0.00,conservar: validar con negocio antes de excluir


## 5. Imputación de valores faltantes
Si una cuenta no tiene movimientos, usamos 0 para los conteos y montos. Para otras variables numéricas se usa la mediana y para las categóricas, la moda. No modificamos la variable Y y revisamos los valores faltantes antes y después.

In [5]:
X=df_limpio[PREDICTORAS].copy()
Y=df_limpio[ETIQUETA].copy()
assert Y.notna().all(), 'No debe imputarse una respuesta sin observar.'
print('Nulos antes de imputación:')
print(X.isna().sum().to_string())

CONTEOS_Y_SUMAS=['num_transacciones','importe_ingresos','importe_egresos']
for col in CONTEOS_Y_SUMAS:
    X[col]=X[col].fillna(0)
for col in NUMERICAS:
    if col not in CONTEOS_Y_SUMAS:
        mediana=X[col].median()
        if pd.isna(mediana): raise ValueError(f'Sin datos para imputar {col}')
        X[col]=X[col].fillna(mediana)
for col in CATEGORICAS:
    moda=X[col].mode(dropna=True)
    if moda.empty: raise ValueError(f'Sin datos para imputar {col}')
    X[col]=X[col].fillna(moda.iloc[0])
print('Nulos después:',int(X.isna().sum().sum()))

Nulos antes de imputación:
edad_1998                    0
antiguedad_cuenta_anios      0
num_transacciones            0
importe_ingresos             0
importe_egresos              0
importe_medio_transaccion    0
saldo_promedio               0
tipo_disposicion             0
frecuencia                   0
Nulos después: 0


## 6. Transformación z-score con `StandardScaler`
Fórmula: \(z=(x-\mu)/\sigma\). Estandarizamos las variables numéricas seleccionadas, pero no los identificadores, las categorías ni Y. En este notebook el cálculo usa la muestra completa.

In [6]:
escalador=StandardScaler()
X[NUMERICAS]=escalador.fit_transform(X[NUMERICAS])
print('Media después del escalado:')
print(X[NUMERICAS].mean().round(4).to_string())
print('Desviación estándar poblacional después:')
print(X[NUMERICAS].std(ddof=0).round(4).to_string())

Media después del escalado:
edad_1998                   -0.0
antiguedad_cuenta_anios      0.0
num_transacciones            0.0
importe_ingresos             0.0
importe_egresos             -0.0
importe_medio_transaccion   -0.0
saldo_promedio              -0.0
Desviación estándar poblacional después:
edad_1998                    1.0
antiguedad_cuenta_anios      1.0
num_transacciones            1.0
importe_ingresos             1.0
importe_egresos              1.0
importe_medio_transaccion    1.0
saldo_promedio               1.0


## 7. Transformaciones one-hot encoding
Como en el notebook del profesor, usamos `pd.get_dummies` y concatenamos las columnas transformadas. Esto evita asignar un orden artificial a las categorías.

In [7]:
columnas_dummies=pd.get_dummies(X[CATEGORICAS],dtype=int)
X_numericas=X.drop(columns=CATEGORICAS)
X_final=pd.concat([X_numericas,columnas_dummies],axis=1)
print('Nuevas columnas one-hot:')
print(columnas_dummies.columns.tolist())
display(X_final.head())

Nuevas columnas one-hot:
['tipo_disposicion_DISPONENT', 'tipo_disposicion_OWNER', 'frecuencia_POPLATEK MESICNE', 'frecuencia_POPLATEK PO OBRATU', 'frecuencia_POPLATEK TYDNE']


,edad_1998,antiguedad_cuenta_anios,num_transacciones,importe_ingresos,importe_egresos,importe_medio_transaccion,saldo_promedio,tipo_disposicion_DISPONENT,tipo_disposicion_OWNER,frecuencia_POPLATEK MESICNE,frecuencia_POPLATEK PO OBRATU,frecuencia_POPLATEK TYDNE
0,-0.972279,0.242834,0.030004,-0.768924,-0.737217,-1.025561,-1.343709,0,1,1,0,0
1,0.474424,1.604891,1.902093,1.313023,1.338031,0.224726,0.004314,0,1,1,0,0
2,0.763764,1.604891,1.902093,1.313023,1.338031,0.224726,0.004314,1,0,1,0,0
3,-0.162126,-1.263355,-0.925623,-0.800481,-0.826192,-0.788663,-0.623368,0,1,1,0,0
4,-0.393598,-1.263355,-0.925623,-0.800481,-0.826192,-0.788663,-0.623368,1,0,1,0,0


## 8. Tabla minable y pruebas de calidad
Conservamos `client_id` para identificar cada fila, pero no como predictor. Excluimos `n_tarjetas`, `fecha_primera_tarjeta` y otros campos que revelan la respuesta. La tabla final tiene una fila por cliente.

In [8]:
tabla_minable=pd.concat([
    df_limpio[['client_id']].reset_index(drop=True),
    X_final.reset_index(drop=True),
    Y.rename('tiene_tarjeta').reset_index(drop=True)],axis=1)

assert len(tabla_minable)==len(df), 'La cantidad de clientes cambió.'
assert tabla_minable['client_id'].is_unique, 'Duplicados por cliente.'
assert tabla_minable.isna().sum().sum()==0, 'Persisten nulos.'
assert tabla_minable['tiene_tarjeta'].isin([0,1]).all()
assert set(tabla_minable['tiene_tarjeta'].unique())==set(df[ETIQUETA].unique())
assert set(['n_tarjetas','fecha_primera_tarjeta','birth_number','disp_id']).isdisjoint(
    tabla_minable.columns), 'Se filtró información directa o identificadores secundarios.'

ruta_csv=RESULTADOS/'04_tabla_minable_berka.csv'
tabla_minable.to_csv(ruta_csv,index=False,encoding='utf-8-sig')
print('Archivo final:',ruta_csv)
print('Filas y columnas:',tabla_minable.shape)
print('Clientes únicos:',tabla_minable['client_id'].nunique())
print('Nulos:',int(tabla_minable.isna().sum().sum()))
display(tabla_minable.head())

Archivo final: /mnt/data/Berka_Equipo/resultados/04_tabla_minable_berka.csv
Filas y columnas: (5369, 14)
Clientes únicos: 5369
Nulos: 0


,client_id,edad_1998,antiguedad_cuenta_anios,num_transacciones,importe_ingresos,importe_egresos,importe_medio_transaccion,saldo_promedio,tipo_disposicion_DISPONENT,tipo_disposicion_OWNER,frecuencia_POPLATEK MESICNE,frecuencia_POPLATEK PO OBRATU,frecuencia_POPLATEK TYDNE,tiene_tarjeta
0,1,-0.972279,0.242834,0.030004,-0.768924,-0.737217,-1.025561,-1.343709,0,1,1,0,0,0
1,2,0.474424,1.604891,1.902093,1.313023,1.338031,0.224726,0.004314,0,1,1,0,0,0
2,3,0.763764,1.604891,1.902093,1.313023,1.338031,0.224726,0.004314,1,0,1,0,0,0
3,4,-0.162126,-1.263355,-0.925623,-0.800481,-0.826192,-0.788663,-0.623368,0,1,1,0,0,0
4,5,-0.393598,-1.263355,-0.925623,-0.800481,-0.826192,-0.788663,-0.623368,1,0,1,0,0,0


## 9. Respuestas a las preguntas de preparación de datos (3.3)
- **Acceso:** los ocho archivos fueron inventariados en el notebook 01; el dataframe se construyó con cinco fuentes.
- **Selección:** se eligieron variables demográficas, de cuenta y de actividad financiera sin usar variables que revelen `Y`.
- **Limpieza:** se registran columnas con >60% de nulos, outliers IQR y reglas de imputación.
- **Integración:** las tablas 1:N se agregaron antes de los `merge`; se verificó unicidad de `client_id`.
- **Herramientas:** usamos Pandas y `sklearn.preprocessing.StandardScaler` para preparar los datos.
- **Formatos:** se normalizaron fechas y categorías; las columnas one-hot son binarias.

La tabla describe la tenencia registrada de tarjetas. Estos datos no indican por sí solos si un cliente es apto para recibir crédito.